# GSB 5544 — Topic 3.2: Distances Between Observations  
*Fill each `____` blank as you work; the ✅ checks ask for a sentence or two, or for you to copy, paste, and edit a cell.*

## The next 15 minutes

You have done the reading, so this is a quick synthesis before PA 3.2 — not a replacement for it.

| | Question | Where it lands in PA 3.2 |
|---|---|---|
| **a. Types** | What does a distance measure, and what options exist? | every part |
| **b. Why** | What question does a distance answer that a filter, a `groupby`, or a join cannot? | Ames 1, College 1 |
| **c. When** | Which variables go in, what to do to them first, and what to leave out | Ames 2–3, College 2–3 |

Last topic combined **two tables** through a key. This topic stays inside **one table** and asks a different question: *which rows are most like this row?* No key, no matching — a **number** that says how far apart two rows are.

In [ ]:
import pandas as pd
import numpy as np

---
## 1. Why distances?  — *"Find me something like this one."*

Questions that sound like this come up constantly, and none of them can be answered with the tools we have so far:

| The question | Filtering / grouping can't, because… |
|---|---|
| *I like house 0 but it is too expensive — find cheaper houses like it.* | "like it" is not a single condition; it is *close on several variables at once* |
| *Which colleges are most similar to Cal Poly?* | there is no category called "Cal-Poly-like" to group by |
| *This customer just signed up — which existing customers do they resemble?* (recommendations; nearest-neighbour prediction later in the course) | we need a **ranking** of all rows by resemblance, not a yes/no |
| *Which transaction looks like none of the others?* (outliers, fraud) | "unlike everything" is *far from every row* |

A **distance** turns "similar" into a number: pick some variables, compute how far each row is from the target row on those variables, and sort. Small distance = similar. The rest of this notebook is about computing that number so that the ranking reflects what *we* mean by "similar" — and not an accident of units.

**The one line of pandas that does it** (you will write it many times in the PA):

```python
dist = np.sqrt(((X - X.loc[target]) ** 2).sum(axis=1))   # Euclidean distance from `target` to every row
```

`X - X.loc[target]` subtracts the target's row from every row (broadcasting); square, sum across the columns (`axis=1`), square-root.

---
## 2. What a distance measures — on a table small enough to check by hand

Three houses, four variables. House **A** is the one we like. Before computing anything, decide for yourself: is **B** (44 more square feet, otherwise identical) or **C** (4 more square feet, but an extra bedroom *and* an extra bathroom) more like A?

In [ ]:
houses = pd.DataFrame({
    "sqft":     [1656, 1700, 1660],
    "bedrooms": [3,    3,    4],
    "baths":    [1,    1,    2],
    "style":    ["1Story", "2Story", "1Story"],
}, index=["A", "B", "C"])
houses

### 2a. Euclidean distance — the default

$$d(i,t) = \sqrt{\sum_j (x_{ij} - x_{tj})^2}$$

Square each difference, add them up, take the square root: the straight-line distance you would measure with a ruler if the variables were axes on a map. This is the distance we use unless there is a reason not to (it is also scikit-learn's default later in the course). Start with the quantitative columns.

In [ ]:
X = houses[["sqft", "bedrooms", "baths"]]

diff = X - X.loc["A"]              # each row minus house A's row
diff

In [ ]:
euclid = np.sqrt((diff ____).sum(axis=____))
euclid

Check by hand: A→B differs only in square feet, so d = √(44²) = **44**. A→C: √(4² + 1² + 1²) = √18 ≈ **4.2**. The formula says C — the house with an extra bedroom *and* bathroom — is ten times closer to A than B, which merely has 44 more square feet. Does that match what you decided above? Probably not. Hold that thought for Section 3.

### 2b. Other options exist — and give different numbers

Euclidean is not the only way to add up differences. **Manhattan distance** adds the absolute differences instead of squaring them:

$$d(i,t) = \sum_j |x_{ij} - x_{tj}|$$

There are others (the quiz asks you to compute a couple). The point is not to memorise them: it is to know that **the choice exists, that different choices give different numbers, and that they can occasionally change which rows come out "closest"**. When in doubt, use Euclidean.

In [ ]:
manhattan = diff.____().sum(axis=1)
pd.DataFrame({"euclidean": euclid, "manhattan": manhattan})

### 2c. A categorical variable — one-hot encode it first

"1Story" minus "2Story" is not a number, so a categorical column cannot go into either formula as it is. **One-hot encode** it: one 0/1 column per category. Two houses with the same style differ by 0 on every style column; two with different styles differ by 1 in two columns — a mismatch adds √2 ≈ 1.41 to a Euclidean distance.

In [ ]:
style = pd.____(houses["style"], dtype=float)
style

In [ ]:
style_dist = np.sqrt(((style - style.loc["A"]) ** 2).sum(axis=1))
style_dist                                   # 0 = same style as A, 1.41 = different style

### 2d. One more option, for *profiles*: cosine similarity

Some rows are **profiles** — the number of students in each field of study, the counts of each word in a document, the mix of products in a basket. For those, we often care about the *mix* and not the *size*. Cosine similarity compares the direction of two rows and ignores their length; it is 1 for an identical mix and 0 for nothing in common (a **similarity**, so *large* means close).

Below, **P, Q, and R are three schools** (the rows); the columns are fields of study, and each cell is the number of students in that field.

In [ ]:
schools = pd.DataFrame({"engineering": [600, 60, 100],
                        "business":    [300, 30, 600],
                        "agriculture": [100, 10, 300]},
                       index=["P", "Q", "R"])            # rows = schools, columns = fields, cells = students
schools

In [ ]:
target = schools.loc["P"]
euclid_schools = np.sqrt(((schools - target) ** 2).sum(axis=1))

norms  = np.sqrt((schools ** 2).sum(axis=1))
cosine = (schools ____ target) / (norms * norms["P"])     # @ = dot product of every row with P

pd.DataFrame({"euclidean": euclid_schools.round(1), "cosine": cosine.round(3)})

**Your answer:** *(write it here — replace this line)*

---
## 3. The two rules before computing any distance

Back to the houses. Square feet are in the thousands; bedrooms and baths are in ones. The raw Euclidean distance added `4²` to `1² + 1²` and decided that an extra bedroom and bathroom matter about as much as four square feet. The **units** made that decision, not us.

> **Rule 1 — always scale the quantitative variables.**
> **Rule 2 — always one-hot encode the categorical variables.**

For Rule 1 we **standardize** (z-scores): subtract each column's mean and divide by its standard deviation, so every column has mean 0 and SD 1 and "one unit" means "one standard deviation" in every column. Other scalings exist — min-max, `(X - X.min()) / (X.max() - X.min())`, squeezes each column into 0–1 — and the PA asks you to try one so you see that the option exists. From then on we always standardize.

In [ ]:
X_z = (X - X.____()) / X.____()
X_z.round(2)

In [ ]:
euclid_z = np.sqrt(((X_z - X_z.loc["A"]) ** 2).sum(axis=1))
pd.DataFrame({"raw euclidean": euclid.round(2), "standardized euclidean": euclid_z.round(2)})

✅ **Check:** which house is nearer to A on the raw scale, and which after standardizing? Which answer matches what you decided at the start of Section 2, and in one sentence, why did it change?

**Your answer:** *(write it here — replace this line)*

### 3b. Quantitative and categorical together

You do not have to choose between them. Standardize the quantitative columns (Rule 1), one-hot encode the categorical one (Rule 2), put the columns side by side in **one** table, and compute **one** distance. PA 3.2 Ames 2 and 3 do exactly this.

In [ ]:
X_all = pd.____([X_z, style], axis=1)     # 3 standardized columns + 3 style indicator columns
X_all.round(2)

In [ ]:
euclid_all = np.sqrt(((X_all - X_all.loc["A"]) ** 2).sum(axis=1))
pd.DataFrame({"standardized (quant only)": euclid_z.round(2), "quant + style": euclid_all.round(2)})

B is a different style from A, so it picks up the √2 mismatch penalty and its distance rises from 1.8 to 2.3; C keeps its 2.5. B is still nearer — but only just. If style should count *less* than a full mismatch, multiply the indicator columns by a weight below 1 (e.g. `0.5 * style`) before concatenating; if it should count *more*, weight it above 1.

---
## 4. Before you compute: three questions

**Question 1 — which variables define "similar" *for this question*?** Distance treats every included column as an equal vote. Choose a few variables deliberately; do not throw in all 80 columns, or ten near-duplicate basement columns will out-vote living area.

**Question 2 — for "cheaper houses like house 0", should `SalePrice` be one of the distance variables?** Think about it before reading on, and be ready to say why.

✅ **Check:** write your answer to Question 2 and your reason in one or two sentences *before* running anything.

**Your answer:** *(write it here — replace this line)*

**Question 3 — which distance, and which scaling?** Rule 1 and Rule 2 are not optional. Beyond that, the table below is a **suggestion, not a rule** — it says what people usually reach for. Different options exist, they are all available to you, and they can influence which rows come out nearest. Standardized Euclidean is the default; the way to find out whether the choice matters for *your* question is to try another one and compare.

| Situation | What people usually do |
|---|---|
| several quantitative variables | standardize, Euclidean |
| a categorical variable in the mix | one-hot encode it, then the same distance |
| rows are profiles (proportions, counts) where only the mix matters | cosine similarity |
| you want one extreme variable to dominate less | Manhattan |

**And always look at the neighbours it picks.** If the "most similar" houses look wrong to a human, the distance is measuring the wrong thing — usually the variable list, not the formula.

---
## 5. The same steps on the real data — PA 3.2 readiness check

The 2,930-house Ames data set the PA uses. The recipe is five steps of ordinary pandas: **select → scale → distance → constrain → sort and look**. Write it once; when the PA asks you to try another option, copy the cell, paste it, and change one line.

In [ ]:
df_ames = pd.read_csv("https://dlsun.github.io/pods/data/AmesHousing.txt", sep="\t")
df_ames["Bathrooms"] = df_ames["Full Bath"] + 0.5 * df_ames["Half Bath"]

house0 = df_ames.loc[0]
house0[["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "House Style", "Neighborhood", "SalePrice"]]

In [ ]:
# 1. select the similarity variables (price is the constraint, so it is NOT here)
X = df_ames[["Gr Liv Area", "Bedroom AbvGr", "Bathrooms"]]

# 2. scale (Rule 1)
X_z = (X - X.mean()) / X.std()

# 3. distance from house 0 to every house
df_ames["dist"] = np.sqrt(((X_z - X_z.____) ** 2).sum(axis=1))

# 4. constrain: only houses cheaper than house 0
cheaper = df_ames[df_ames["SalePrice"] ____ house0["SalePrice"]]

# 5. sort and look
show = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "House Style", "Neighborhood", "SalePrice", "dist"]
cheaper.sort_values("dist")[show].head(5)

✅ **Check (copy, paste, edit):** copy the cell above into the empty cell below and change **one** line so it uses Manhattan distance (absolute differences, no square root). Then copy it again and *remove* the scaling step (use `X` instead of `X_z`). Which change alters the five houses?

In [ ]:
# Manhattan: only step 3 changes
X = df_ames[["Gr Liv Area", "Bedroom AbvGr", "Bathrooms"]]
X_z = (X - X.mean()) / X.std()
df_ames["dist"] = (X_z - X_z.loc[0]).____().sum(axis=1)
cheaper = df_ames[df_ames["SalePrice"] < house0["SalePrice"]]
cheaper.sort_values("dist")[show].head(5)

In [ ]:
# No scaling: step 2 removed, step 3 uses X instead of X_z
X = df_ames[["Gr Liv Area", "Bedroom AbvGr", "Bathrooms"]]
df_ames["dist"] = np.sqrt(((____ - ____.loc[0]) ** 2).sum(axis=1))
cheaper = df_ames[df_ames["SalePrice"] < house0["SalePrice"]]
cheaper.sort_values("dist")[show].head(5)

**Your answer:** *(write it here — replace this line)*

---
## 6. Before you start the PA — pair up

With the person next to you, agree on answers to these three questions for PA 3.2 Ames part 3, where *you* choose the variables. Be ready to share with the class.

1. Which **four to six** variables would you use to decide that two houses are "similar"? Name at least one categorical variable.
2. Which variable(s) are **constraints** rather than similarity variables?
3. Which *one* option would you change (scaling, formula, or the variable list) to test whether your answer is robust — and what would convince you it matters?

## Summary

| | |
|---|---|
| **What a distance is** | one number per row saying how far it is from a target row on chosen variables; small = similar |
| **Why** | "find rows like this one": recommendations, comparable cases, nearest-neighbour prediction, outliers — questions with no category to filter or group on |
| **Options** | Euclidean is the default; Manhattan, cosine similarity, and others exist and can give different answers — try one to see whether it matters |
| **Rule 1** | always scale the quantitative variables (we standardize) |
| **Rule 2** | always one-hot encode the categorical variables, then put everything in one table |
| **Before computing** | pick the similarity variables deliberately; keep constraints (like price) out of the distance and filter on them afterwards |
| **After computing** | sort, **look** at the neighbours, and change one option to check sensitivity |
| **The code** | `X_z = (X - X.mean()) / X.std()` → `np.sqrt(((X_z - X_z.loc[t]) ** 2).sum(axis=1))` → filter → `.sort_values("dist").head(k)` |

PA 3.2 is on the course site: [https://gato365.github.io/gsb5544_instructor_learn_prep/](https://gato365.github.io/gsb5544_instructor_learn_prep/).